<a href="https://colab.research.google.com/github/julianvanegas/DB-Structure/blob/main/Descifrar_hash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import hashlib
import time
from itertools import product, permutations

## 1. Encontrar la secuencia de 10 números enteros entre [0,9] que generan un hash SHA-256

In [52]:
def find_sequence(target_hash):
    """Busca una secuencia numérica de 10 dígitos cuyo SHA-256 coincida con target_hash."""
    start_time = time.time()
    iterations = 0

    for combo in product(range(10), repeat=10):
      # Concatenar dígitos
      sequence = "".join(map(str, combo))
      current_hash = hashlib.sha256(sequence.encode()).hexdigest()

      iterations += 1

      if current_hash == target_hash:
        print(f"\n  Secuencia: {sequence}")
        print(f"   Intentos: {iterations}")
        print(f"   Tiempo: {time.time() - start_time:.2f}s")
        return sequence

    print("No se encontró la secuencia.")
    return None


In [53]:
hash = hashlib.sha256("0123456789".encode()).hexdigest()
print(f"Por ejemplo el hash de '0123456789': {hash}\n")

hash = input("Ingrese el hash a descifrar: ")
find_sequence(hash)

Por ejemplo el hash de '0123456789': 84d89877f0d4041efb6bf91a16f0248f2fd573e6af05c19f96bedb9f882f7882

Ingrese el hash a descifrar: 84d89877f0d4041efb6bf91a16f0248f2fd573e6af05c19f96bedb9f882f7882

  Secuencia: 0123456789
   Intentos: 123456790
   Tiempo: 341.80s


'0123456789'

## 2. Encontrar el orden de transacciones para la reíz de un árbol de Merkle

In [60]:
class merkle_tree:

    def __init__(self, transactions):
      self._transactions = transactions
      self._root, self._levels = self.tree()

    def hash(self, transaction):
      """Calcula el hash de un string con SHA-256."""
      h = hashlib.sha256(str(transaction).encode('utf-8'))
      return h.hexdigest()

    def level(self, hashes):
      """Un nivel del árbol combina pares de hashes."""
      if len(hashes) % 2 == 0:
        return [self.hash(f"{hashes[i]}{hashes[i+1]}") for i in range(0, len(hashes), 2)]
      else:
        pairs = [self.hash(f"{hashes[i]}{hashes[i+1]}") for i in range(0, len(hashes) - 1, 2)]
        pairs.append(hashes[-1])
        return pairs

    def tree(self):
      """Construye el árbol de abajo hacia arriba. Retorna la raíz y todos los niveles."""
      if not self._transactions:
        return None, []
      # Nivel 0: hashear las transacciones originales
      level0 = [self.hash(t) for t in self._transactions]

      if len(level0) == 1:
        return level0[0], [level0]

      levels = [level0]
      current = level0

      while len(current) > 1:
        current = self.level(current)
        levels.append(current)

      root = current[0]
      return root, levels

    def show(self):
      depth = len(self._levels)

      print(f"Raíz: {self._root}")

      for i in range(depth - 2, -1, -1):
        level = self._levels[i]
        level_name = f"Nivel {i}"
        display = [node for node in level]
        indent = " " * ((depth - 1 - i) * 2)
        nodes_str = " | ".join(display)
        print(f"{indent}{level_name}: [ {nodes_str} ]")


In [61]:
def find_order(target_root, transactions):
  start_time = time.time()
  iterations = 0

  # permutations genera todas las combinaciones posibles de orden
  for p in permutations(transactions):
    iterations += 1
    # Calculamos el root con el orden actual
    current_root = merkle_tree(list(p))._root

    if current_root == target_root:
      print(f"\nORDEN ENCONTRADO")
      print(f"  Orden: {list(p)}")
      print(f"  Intentos: {iterations}.")
      print(f"  Tiempo: {time.time() - start_time:.2f}s")
      return list(p)

  print("No se encontró ningún orden que genere el root.")
  return None


In [62]:
transactions = ["TXe", "TXc", "TXd", "TXa", "TXb"] # Transacciones para generar el árbol.
tree = merkle_tree(transactions)
tree.show()

Raíz: 6facd3f15b337c2b42ab1e06350816b3818d8d02f5f46c7efdf76f2d65422d25
  Nivel 2: [ 76443d2935674719ef803739a8c1f253cff3d0d485cb821db8d23a8ce34ef920 | 334cfaa4acbd199119c5f342bc4faec946c66aeb5694fc0cb13a4775585a1f40 ]
    Nivel 1: [ 760fca7e140ad8844f2d0f1f5d593577740513c4b601ae54aefe53600c385bfc | 5b61d1c7449f4f84a8c66e51b64be4c761b507012c9664508c32b8f2849d380a | 334cfaa4acbd199119c5f342bc4faec946c66aeb5694fc0cb13a4775585a1f40 ]
      Nivel 0: [ 87cdbdbabd3db8069a00eb02a6d054ac05456b4557e7bce8958f8a0347d945d7 | 13cedf2ee04f73b0c8d8b053d4abfed7b7730a0fd6bd99d5cddb84b046c3b30b | 1422781f65d3f95899fd9090ef7ebeb5359b2925c566017037b7f93e6621c129 | 6398122affa6b4869b7a3799f63d23ffe7406dd912395fa117f37927af81529b | 334cfaa4acbd199119c5f342bc4faec946c66aeb5694fc0cb13a4775585a1f40 ]


In [63]:
transactions = ["TXa", "TXb", "TXc", "TXd", "TXe"] # Transacciones en desorden con las que se generó el árbol.
find_order(tree._root, transactions)


ORDEN ENCONTRADO
  Orden: ['TXe', 'TXc', 'TXd', 'TXa', 'TXb']
  Intentos: 113.
  Tiempo: 0.00s


['TXe', 'TXc', 'TXd', 'TXa', 'TXb']